# Production Pipeline

> **Source:** `repo2/02_contextual_retrieval.py`

Demonstrate a production-ready contextual retrieval pipeline.


## Imports and Setup


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import time
import tiktoken
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
load_dotenv()


## Implementation


In [ ]:
def demo_production_pipeline():
    """
    Demonstrate a production-ready contextual retrieval pipeline.
    """

    print("\n" + "=" * 60)
    print("PRODUCTION PIPELINE DEMO")
    print("=" * 60)

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    # Sample documents
    documents = [
        Document(
            page_content="""
            TechStartup Inc. Series B Funding Announcement

            TechStartup Inc., a leading AI infrastructure company based in Seattle,
            today announced the closing of its Series B funding round. The round
            raised $45 million, led by Sequoia Capital with participation from
            Andreessen Horowitz.

            The company plans to use the funds to expand its engineering team and
            accelerate product development. CEO Jane Smith stated that the company
            expects to double its headcount by end of 2026.

            TechStartup's flagship product, AIFlow, helps enterprises deploy and
            manage large language models in production. The platform currently
            serves over 200 enterprise customers.
            """,
            metadata={
                "title": "TechStartup Inc. Series B Announcement",
                "source": "press_release.pdf",
            },
        )
    ]

    print("\nStep 1: Creating contextualized chunks...")
    contextualized_docs = create_contextual_chunks(
        documents=documents, llm=llm, chunk_size=200, chunk_overlap=50
    )

    print(f"Created {len(contextualized_docs)} contextualized chunks")

    print("\nStep 2: Creating vector store...")
    vectorstore = Chroma.from_documents(
        documents=contextualized_docs,
        embedding=embeddings,
        collection_name="contextual_demo",
    )

    print("\nStep 3: Testing retrieval...")
    query = "How much funding did the Seattle AI company raise?"

    results = vectorstore.similarity_search(query, k=2)

    print(f'\nQuery: "{query}"')
    print("\nTop results:")
    for i, doc in enumerate(results, 1):
        print(f"\n  Result {i}:")
        print(f"    Content: {doc.page_content[:100]}...")
        print(
            f"    Context prefix: {doc.metadata.get('context_prefix', 'N/A')[:50]}..."
        )

    # Cleanup
    vectorstore.delete_collection()

    print("\n" + "-" * 60)
    print("PRODUCTION CONSIDERATIONS:")
    print("-" * 60)
    print(
        """
    1. COST: ~$0.01-0.05 per document for context generation
       - One-time cost at indexing time
       - Much cheaper than retrieval failures

    2. LATENCY: Adds 1-2s per chunk during indexing
       - Batch process documents offline
       - No impact on query latency

    3. STORAGE: Chunks are ~20-30% larger
       - Minimal impact on vector DB costs

    4. WHEN TO USE:
       - Documents have important context in headers/titles
       - Entities are referenced with pronouns ("the company", "they")
       - Documents come from multiple sources
    """
    )


## Execute Demo


In [ ]:
demo_production_pipeline()
